# Yoga Pose Classification — Training Notebook

This notebook replicates the full pipeline: keypoint extraction → training → evaluation.
Designed to run in Google Colab with a free T4 GPU.

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install dependencies
!pip install torch torchvision mediapipe opencv-python scikit-learn matplotlib seaborn tensorboard tqdm pyyaml

In [ ]:
# Mount Drive (optional, for saving checkpoints)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import sys
import torch
import numpy as np
from pathlib import Path

# Add src to path
sys.path.insert(0, 'src')

from src.data.keypoint_extraction import MediaPipeKeypointExtractor
from src.data.dataset import get_dataloaders
from src.models.cross_modal_attention import CrossModalPoseClassifier, BaselineMLP
from src.training.trainer import Trainer
from src.evaluation.evaluator import Evaluator

## 1. Upload / Download Dataset

Upload your dataset to `data/raw/` with one subdirectory per class.
Or run the Kaggle download cell below.

In [ ]:
# Example: manual upload or Kaggle download
# !kaggle datasets download -d niharika41298/yoga-poses-dataset -p data/raw --unzip

## 2. Extract Keypoints with MediaPipe

In [ ]:
RAW_DIR = 'data/raw'
KP_DIR = 'data/processed/keypoints'

extractor = MediaPipeKeypointExtractor()

for pose_class in sorted(os.listdir(RAW_DIR)):
    class_dir = os.path.join(RAW_DIR, pose_class)
    if not os.path.isdir(class_dir):
        continue
    out_dir = os.path.join(KP_DIR, pose_class)
    result = extractor.extract_from_directory(class_dir, out_dir, label=pose_class)
    print(f"{pose_class}: {result['extracted']}/{result['total']} extracted")

## 3. Create DataLoaders

In [ ]:
BATCH_SIZE = 64
NUM_WORKERS = 2

train_loader, val_loader, test_loader, class_names = get_dataloaders(
    data_dir=KP_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)
NUM_CLASSES = len(class_names)
print(f"Classes: {NUM_CLASSES}")
print(f"Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)} | Test: {len(test_loader.dataset)}")

## 4. Train Cross-Modal Attention Model

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = CrossModalPoseClassifier(
    num_classes=NUM_CLASSES,
    d_model=128,
    hidden_dim=64,
    num_attention_layers=2,
    dropout=0.3,
)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_classes=NUM_CLASSES,
    device=device,
    lr=1e-3,
    weight_decay=1e-4,
    max_epochs=50,
    patience=10,
    model_name='cross_modal',
)

history = trainer.fit()

## 5. Evaluate

In [ ]:
evaluator = Evaluator(
    model=model,
    test_loader=test_loader,
    class_names=class_names,
    device=device,
)
results = evaluator.evaluate()
print(results)

## 6. Train Baseline MLP (for comparison)

In [ ]:
baseline = BaselineMLP(
    input_dim=68,
    hidden_dims=[256, 128],
    num_classes=NUM_CLASSES,
    dropout=0.3,
)

baseline_trainer = Trainer(
    model=baseline,
    train_loader=train_loader,
    val_loader=val_loader,
    num_classes=NUM_CLASSES,
    device=device,
    lr=1e-3,
    weight_decay=1e-4,
    max_epochs=50,
    patience=10,
    model_name='baseline',
)

baseline_history = baseline_trainer.fit()

baseline_evaluator = Evaluator(
    model=baseline,
    test_loader=test_loader,
    class_names=class_names,
    device=device,
)
baseline_results = baseline_evaluator.evaluate()

## 7. Save Models to Drive (optional)

In [ ]:
# !cp -r outputs/checkpoints /content/drive/MyDrive/yoga_pose_classifier/
# !cp -r outputs/figures /content/drive/MyDrive/yoga_pose_classifier/